In [ ]:
"""Score test-set carts, attach plain-English reasons and brand, write a SQLite DB for the dashboard.

Memory-friendly version: large CSV files are read in chunks and filtered to the test rows.
Run from the notebooks folder:  python export_scored_carts.py
"""
import json
import sqlite3
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

DATA_DIR = Path("../data")
ARTIFACT_DIR = Path("../artifacts")
SCORE_CHUNK = 100_000     # rows scored per batch
READ_CHUNK = 300_000      # rows read from CSV per batch
MIN_REASON_SCORE = 0.05   # a reason must contribute at least this much (log-odds) to be shown

# ---------------------------------------------------------------------------
# Reason catalog: what the store owner sees
# ---------------------------------------------------------------------------
REASON_CATALOG = {
    "CART_HESITATION": ("Hesitation in this session",
        "The customer already removed items from their cart during this visit, which usually signals indecision.",
        "Show a reminder or a small incentive, such as a free-shipping threshold."),
    "HABITUAL_REMOVER": ("Uses the cart like a wishlist",
        "This customer often adds items to the cart and then removes them.",
        "Offer a 'Save for later' list and price-drop alerts."),
    "LOW_BUYER_HISTORY": ("Rarely completes purchases",
        "This customer has a low purchase rate compared with how often they add items to the cart.",
        "Offer a first-purchase incentive and show trust signals such as reviews and the return policy."),
    "HEAVY_BROWSING_HISTORY": ("Browses a lot, buys little",
        "Past activity shows many views and cart additions relative to purchases.",
        "Highlight best-sellers and limited-time offers to shorten the decision time."),
    "WEAK_PRODUCT_CONVERSION": ("Product rarely sells",
        "Few customers who add this product to their cart go on to buy it.",
        "Review the product page, photos, reviews, price and stock availability."),
    "WEAK_CATEGORY_CONVERSION": ("Category converts poorly",
        "Carts in this category are often abandoned or cleared.",
        "Check category pricing, delivery options and competing alternatives."),
    "PRODUCT_REVISITED_NO_BUY": ("Interested in this product but not buying",
        "The customer viewed or added this exact product before without purchasing it.",
        "Send a targeted reminder or a time-limited discount for this product."),
    "PRICE_BARRIER": ("Price may be a barrier",
        "The price is high compared with the category average or the product's usual price.",
        "Test a discount, a bundle or an installment option."),
    "PRICE_CHANGE": ("Price changed since last view",
        "The price differs from what the customer saw when they last viewed the product.",
        "Communicate price changes clearly and consider price-match messaging."),
    "LARGE_CART": ("Large cart",
        "The cart holds many items or a high total value, which increases hesitation.",
        "Offer free shipping, payment plans or money-back guarantee badges."),
    "COMPARISON_SHOPPING": ("Comparing alternatives",
        "The customer looked at many products, categories or brands before adding to the cart.",
        "Show comparison tools, similar items and clear differentiators."),
    "LOW_INTENT_SESSION": ("Low purchase intent in this session",
        "Session behaviour suggests casual interest, for example adding to cart without browsing.",
        "Use exit-intent messages and email or push reminders."),
    "VISIT_PATTERN": ("Visit pattern",
        "The customer is new or returns after a long gap between visits.",
        "Use welcome offers and re-engagement emails."),
    "TIMING": ("Time of day or week",
        "The cart was created at a time when abandonment is more common.",
        "Schedule reminders and promotions around these hours."),
    "OTHER": ("Other factors",
        "A mix of smaller factors.",
        "Review the customer's behaviour in more detail."),
}

EXPLICIT_MAP = {
    "removes_before": "CART_HESITATION",
    "net_cart_value": "LARGE_CART", "cart_items_net": "LARGE_CART",
    "price": "PRICE_BARRIER", "category_avg_price": "PRICE_BARRIER",
    "price_vs_category_avg": "PRICE_BARRIER", "price_ratio_to_category": "PRICE_BARRIER",
    "price_ratio_to_product_median": "PRICE_BARRIER",
    "price_change_since_view": "PRICE_CHANGE",
    "views_before_cart": "COMPARISON_SHOPPING", "category_diversity_before_cart": "COMPARISON_SHOPPING",
    "brand_switch_count_before_cart": "COMPARISON_SHOPPING", "hours_since_last_view": "COMPARISON_SHOPPING",
    "has_prior_view_of_product": "COMPARISON_SHOPPING",
    "no_view_before_cart": "LOW_INTENT_SESSION", "is_first_cart_in_session": "LOW_INTENT_SESSION",
    "session_age_min": "LOW_INTENT_SESSION", "secs_since_prev_event": "LOW_INTENT_SESSION",
    "events_in_session_before": "LOW_INTENT_SESSION", "carts_before": "LOW_INTENT_SESSION",
    "purchases_before_in_session": "LOW_INTENT_SESSION",
    "previous_purchase_count": "LOW_BUYER_HISTORY", "has_previous_purchase": "LOW_BUYER_HISTORY",
    "previous_cart_events": "HEAVY_BROWSING_HISTORY",
    "previous_sessions": "VISIT_PATTERN", "hours_since_prev_session": "VISIT_PATTERN",
    "days_since_first_event": "VISIT_PATTERN",
    "cart_hour": "TIMING", "cart_day_of_week": "TIMING", "cart_is_weekend": "TIMING",
    "cart_day_of_year": "TIMING",
}


def feature_to_reason(col: str) -> str:
    if col in EXPLICIT_MAP:
        return EXPLICIT_MAP[col]
    if col.startswith("userprod_"):
        return "PRODUCT_REVISITED_NO_BUY"
    if col.startswith("prod_"):
        return "WEAK_PRODUCT_CONVERSION"
    if col.startswith("cat_"):
        return "WEAK_CATEGORY_CONVERSION"
    if col.startswith("user_"):
        if "remove" in col:
            return "HABITUAL_REMOVER"
        if "buy_rate" in col or "purchases" in col:
            return "LOW_BUYER_HISTORY"
        return "HEAVY_BROWSING_HISTORY"
    return "OTHER"


# ---------------------------------------------------------------------------
# Memory-friendly CSV readers
# ---------------------------------------------------------------------------
def read_rows(path, usecols, row_ids):
    """Read a big CSV in chunks and keep only the rows whose row_id is in `row_ids`."""
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=READ_CHUNK):
        keep = np.isin(chunk["row_id"].to_numpy(), row_ids)
        parts.append(chunk[keep])
    return pd.concat(parts, ignore_index=True)


def read_brand(path, row_ids):
    """Brand info for the given row positions (row_id = row position in reconstructed_events.csv)."""
    parts = []
    start = 0
    for chunk in pd.read_csv(path, usecols=["brand", "brand_imputed"], chunksize=READ_CHUNK):
        pos = np.arange(start, start + len(chunk))
        keep = np.isin(pos, row_ids)
        part = chunk[keep].copy()
        part.insert(0, "row_id", pos[keep])
        parts.append(part)
        start += len(chunk)
    return pd.concat(parts, ignore_index=True)


def main():
    meta = json.loads((ARTIFACT_DIR / "metadata.json").read_text())
    features = meta["features"]
    band_low, band_high = meta["band_low"], meta["band_high"]

    model = xgb.XGBClassifier()
    model.load_model(str(ARTIFACT_DIR / "model.json"))
    booster = model.get_booster()
    calibrator = joblib.load(ARTIFACT_DIR / "calibrator.joblib")
    test_row_ids = np.load(ARTIFACT_DIR / "test_row_ids.npy")

    # ---- Features for test rows only (out-of-sample scoring) ----
    print("Reading features (chunked) ...")
    feats = read_rows(DATA_DIR / "final_model_ready_cart_events_v2.csv",
                      ["row_id"] + features, test_row_ids)
    X = feats[features].astype("float32")
    row_ids = feats["row_id"].to_numpy()
    del feats
    n = len(X)
    print("Test carts to score:", n)

    # ---- Labels and cart metadata ----
    print("Reading labels (chunked) ...")
    labels = read_rows(DATA_DIR / "cart_labels_v2.csv",
                       ["row_id", "user_id", "product_id", "category_id",
                        "price", "event_time", "outcome", "is_abandoned"], test_row_ids)
    scored = pd.DataFrame({"row_id": row_ids}).merge(labels, on="row_id", how="left",
                                                      validate="one_to_one")
    del labels

    # ---- Brand ----
    print("Reading brand (chunked) ...")
    brand_df = read_brand(DATA_DIR / "reconstructed_events.csv", test_row_ids)
    brand_df["brand"] = brand_df["brand"].astype(object).fillna("unknown")
    brand_df["brand_is_real"] = (~brand_df["brand_imputed"].astype(bool)).astype(int)
    scored = scored.merge(brand_df[["row_id", "brand", "brand_is_real"]], on="row_id",
                          how="left", validate="one_to_one")
    scored["brand"] = scored["brand"].fillna("unknown")
    scored["brand_is_real"] = scored["brand_is_real"].fillna(0).astype(int)
    del brand_df
    assert (scored["row_id"].to_numpy() == row_ids).all(), "row order changed during merges"

    # ---- Score + SHAP contributions, in chunks ----
    codes = list(REASON_CATALOG)
    code_arr = np.array(codes, dtype=object)
    M = np.zeros((len(features), len(codes)), dtype="float32")   # feature -> reason matrix
    for i, f in enumerate(features):
        M[i, codes.index(feature_to_reason(f))] = 1.0

    raw = np.empty(n, dtype="float32")
    reason_scores = np.zeros((n, len(codes)), dtype="float32")
    for start in range(0, n, SCORE_CHUNK):
        end = min(start + SCORE_CHUNK, n)
        dm = xgb.DMatrix(X.iloc[start:end], feature_names=features)
        contrib = booster.predict(dm, pred_contribs=True)          # last column is the bias term
        margin = contrib.sum(axis=1)
        raw[start:end] = 1.0 / (1.0 + np.exp(-margin))
        reason_scores[start:end] = np.clip(contrib[:, :-1], 0, None) @ M
        print(f"  scored {end:,} / {n:,}")
    del X

    risk = calibrator.predict(raw)
    band = np.where(risk >= band_high, "High", np.where(risk >= band_low, "Medium", "Low"))

    top_idx = np.argsort(-reason_scores, axis=1)[:, :3]
    top_val = np.take_along_axis(reason_scores, top_idx, axis=1)
    for k in range(3):
        col = np.where((top_val[:, k] >= MIN_REASON_SCORE) & (band != "Low"),
                       code_arr[top_idx[:, k]], "")
        scored[f"reason_{k + 1}"] = col
    scored["main_reason_code"] = scored["reason_1"]

    scored["risk_score"] = risk.round(4)
    scored["risk_band"] = band
    scored["actual_is_abandoned"] = scored["is_abandoned"].astype(int)
    scored["actual_outcome"] = scored["outcome"]
    ts = pd.to_datetime(scored["event_time"], utc=True)
    scored["event_time"] = ts.dt.strftime("%Y-%m-%d %H:%M:%S")
    scored["event_date"] = ts.dt.strftime("%Y-%m-%d")
    scored = scored.drop(columns=["is_abandoned", "outcome"])

    # ---- Write outputs ----
    catalog = pd.DataFrame([{"code": c, "title": t, "explanation": e, "suggested_action": a}
                            for c, (t, e, a) in REASON_CATALOG.items()])
    catalog.to_csv(ARTIFACT_DIR / "reasons_catalog.csv", index=False)
    scored.to_csv(ARTIFACT_DIR / "scored_carts.csv", index=False)

    db_path = ARTIFACT_DIR / "dashboard.db"
    if db_path.exists():
        db_path.unlink()
    with sqlite3.connect(db_path) as con:
        scored.to_sql("scored_carts", con, index=False, chunksize=50_000)
        catalog.to_sql("reasons_catalog", con, index=False)
        for col in ["brand", "category_id", "risk_band", "event_date", "main_reason_code"]:
            con.execute(f"CREATE INDEX idx_scored_{col} ON scored_carts({col})")

    # ---- Sanity report ----
    print("\nRows written:", len(scored))
    print("\nRisk band share and actual abandonment rate:")
    print(scored.groupby("risk_band")["actual_is_abandoned"].agg(["size", "mean"]).round(3))
    print("\nMain reasons among High-risk carts (%):")
    high = scored[scored["risk_band"] == "High"]
    print(high["main_reason_code"].value_counts(normalize=True).mul(100).round(1))
    print("\nTop 10 brands by revenue at risk (High band):")
    print(high.groupby("brand")["price"].agg(["size", "sum"]).sort_values("sum", ascending=False).head(10).round(0))
    print("\nSaved:", db_path.resolve())


if __name__ == "__main__":
    main()

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

db = Path("../artifacts/dashboard.db")
print("exists:", db.exists(), "| size MB:", round(db.stat().st_size / 1e6, 1) if db.exists() else None)

with sqlite3.connect(db) as con:
    print(pd.read_sql("SELECT COUNT(*) AS total_rows FROM scored_carts", con))

    print(pd.read_sql("""
        SELECT risk_band, COUNT(*) AS carts, ROUND(AVG(actual_is_abandoned), 3) AS actual_abandon_rate
        FROM scored_carts GROUP BY risk_band
    """, con))

    print(pd.read_sql("""
        SELECT main_reason_code, COUNT(*) AS carts
        FROM scored_carts WHERE risk_band = 'High'
        GROUP BY main_reason_code ORDER BY carts DESC
    """, con))

    print(pd.read_sql("""
        SELECT brand, COUNT(*) AS carts, ROUND(SUM(price)) AS revenue_at_risk
        FROM scored_carts WHERE risk_band = 'High'
        GROUP BY brand ORDER BY revenue_at_risk DESC LIMIT 10
    """, con))

exists: True | size MB: 151.1
   total_rows
0      630303
  risk_band   carts  actual_abandon_rate
0      High  170457                0.886
1       Low  245038                0.575
2    Medium  214808                0.771
            main_reason_code  carts
0    WEAK_PRODUCT_CONVERSION  46115
1           HABITUAL_REMOVER  36219
2              VISIT_PATTERN  34715
3   PRODUCT_REVISITED_NO_BUY  11686
4   WEAK_CATEGORY_CONVERSION   8284
5                     TIMING   7273
6                 LARGE_CART   6968
7            CART_HESITATION   6350
8          LOW_BUYER_HISTORY   5973
9        COMPARISON_SHOPPING   3012
10    HEAVY_BROWSING_HISTORY   1675
11        LOW_INTENT_SESSION   1632
12             PRICE_BARRIER    555
      brand  carts  revenue_at_risk
0   unknown  37880         158321.0
1    runail  13939          73089.0
2     irisk  14049          65552.0
3    strong    303          58010.0
4   grattol   7183          42082.0
5  jessnail   1398          37966.0
6    masura  11192    